# Администрирование Cloudberry / Greenplum — 30 заданий

Модуль учит диагностировать кластер без опасных изменений. Любые cancel/terminate/vacuum выполняются только на собственной учебной сессии или таблице.

## Результаты обучения

После **Эксплуатация Greenplum** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

Кластер считается здоровым, когда доступны coordinator, segments и interconnect, нет skew/долгих блокировок, статистика и bloat контролируются, а recovery проверен.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

системные views gp_segment_configuration, pg_stat_activity и catalog. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Соберите наблюдение, порог, диагноз и безопасное действие. Runbook обязан содержать проверку до и после, а не только команду.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 200
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Область ответственности

Администрирование включает доступность, производительность, ёмкость, безопасность и восстановление. Диагностика начинается с наблюдения и сбора evidence; случайный restart или terminate может скрыть причину и усугубить инцидент.

## 2. Топология

Content — логический сегмент. Primary обслуживает запросы, mirror синхронно поддерживает копию. Standby защищает coordinator. `role` — текущая роль, `preferred_role` — исходная. Status `u` и mode `s` — нормальное синхронное состояние.

## 3. Fault domains

Primary и его mirror должны находиться на разных host/failure domains. Баланс primary по host влияет на нормальную нагрузку и состояние после failover.

## 4. Failover

При недоступности primary mirror может получить роль primary. После восстановления требуется вернуть redundancy/resync. Учебный курс не инициирует failover автоматически, потому что контейнеры содержат локальное состояние.

## 5. Сессия, запрос, транзакция

`pg_stat_activity` различает backend session, текущий query и transaction. Долгий query может быть нормальным ETL; idle in transaction бездействует, но удерживает snapshot/locks. Сравнивайте query_start и xact_start.

## 6. Блокировки

Blocking — нормальный механизм согласованности, пока ожидание ограничено. Диагностика строит цепочку blocked→blocker, тип/объект lock и возраст транзакции. Удаляют причину, а не обязательно самую заметную жертву.

## 7. Cancel и terminate

Cancel просит отменить текущий statement, сохраняя session. Terminate завершает backend и откатывает транзакцию. Сначала проверяют PID, user, query, transaction и downstream impact. Никогда не применяйте к неизвестной сессии по одному возрасту.

## 8. Размер

Логический размер таблицы распределён по сегментам. Partition parent может почти не иметь данных, а leaves — занимать весь объём. Не суммируйте parent и leaves одновременно. Отдельно учитывайте indexes/AO auxiliary.

## 9. Skew

Row skew влияет на CPU/число операций, byte skew — на I/O/ёмкость. Следите за трендом: растущий skew может появиться из-за изменения данных без изменения DDL.

## 10. Heap bloat

MVCC оставляет dead tuples до vacuum. `VACUUM` делает место повторно используемым, но обычно не возвращает файл ОС. Rewrite/VACUUM FULL намного тяжелее и требует отдельного планирования.

## 11. AO maintenance

AO использует segment files, visibility map и auxiliary metadata. Частые UPDATE/DELETE создают невидимые строки. Диагностика AO отличается от heap; не применяйте heap-only формулу bloat.

## 12. Statistics

Отсутствующая/устаревшая статистика ухудшает cardinality и план. ANALYZE запускают после существенной загрузки и отслеживают нужные колонки. Cluster-wide analyze может быть дорогим.

## 13. Resource management

Resource groups ограничивают concurrency, CPU и memory. Statement memory влияет на hash/sort/aggregate, но является частью общего бюджета. Увеличение одной сессии может ухудшить кластер для остальных.

## 14. Spill

Workfile создаётся, когда оператор не помещается в память. Небольшой spill допустим; массовый spill замедляет запрос и заполняет диск. Лечение: статистика, уменьшение/проекция данных, physical design, join order или обоснованная память.

## 15. Capacity

Следят за ростом данных, свободным местом каждого host, количеством files/partitions, временными файлами и HDFS. Среднее свободное место скрывает переполненный segment host.

## 16. PXF/HDFS

SQL extension может быть установлено, но PXF server остановлен или Hadoop config/path недоступны. Проверка идёт слоями: extension→server 5888 на обоих hosts→NameNode→DataNode→permissions/path→format.

## 17. Инцидент

Сначала timestamp/scope/symptom, затем cluster health, blockers, plan, actual rows/skew, Motion, spill, statistics и recent changes. После mitigation сохраняют root cause и preventive action.

## 18. Runbook

Runbook содержит безопасные read-only проверки, критерии решений, команды с областью действия, rollback и момент эскалации. Команда без предусловий — не runbook.

## 19. Мониторинг

Dashboard показывает состояние, но alert должен быть actionable. Метрики: unavailable instances, replication mode, query age, idle tx, blockers, storage/skew, stats gaps, spill, PXF/HDFS.

## 20. Безопасность упражнений

Не останавливайте сегменты, не удаляйте data directories, не меняйте cluster configs и не завершайте чужие backend. Все destructive experiments ограничиваются `greenplum_training.admin_lab` и собственными сессиями.

### Задание 1. `m_razhin.gpa_01_topology`

**Что сделать:** Создайте VIEW полной топологии content/primary/mirror/standby.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

gp_segment_configuration.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_01_topology.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',1);

### Задание 2. `m_razhin.gpa_02_health`

**Что сделать:** Создайте сводку up/down и sync mode.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Нормальное состояние: все u, mirrors s.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_02_health.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',2);

### Задание 3. `m_razhin.gpa_03_role_drift`

**Что сделать:** Найдите несовпадение role/preferred_role.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

После failover роли могут отличаться.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_03_role_drift.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',3);

### Задание 4. `m_razhin.gpa_04_host_balance`

**Что сделать:** Покажите число primary/mirror по host.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Fault domains должны быть разнесены.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_04_host_balance.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',4);

### Задание 5. `m_razhin.gpa_05_version`

**Что сделать:** Создайте VIEW версии и ключевых extensions.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

PXF/gp_toolkit обязательны для курса.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_05_version.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',5);

### Задание 6. `m_razhin.gpa_06_sessions`

**Что сделать:** Создайте безопасный VIEW активных сессий.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Не показывайте пароль/секреты query text.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_06_sessions.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',6);

### Задание 7. `m_razhin.gpa_07_long_queries`

**Что сделать:** Найдите активные запросы дольше порога.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Различайте query_start и xact_start.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_07_long_queries.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',7);

### Задание 8. `m_razhin.gpa_08_idle_tx`

**Что сделать:** Найдите idle in transaction.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Они удерживают snapshot/locks.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_08_idle_tx.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',8);

### Задание 9. `m_razhin.gpa_09_locks`

**Что сделать:** Создайте VIEW ожидающих блокировки.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Свяжите pg_locks и activity.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_09_locks.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',9);

### Задание 10. `m_razhin.gpa_10_blockers`

**Что сделать:** Постройте blocked→blocker mapping.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Используйте blocking pids/lock keys.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_10_blockers.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',10);

## Уровень 2 — storage, skew и maintenance

### Задание 11. `m_razhin.gpa_11_schema_sizes`

**Что сделать:** Суммируйте размер таблиц по schema.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Partition leaves учитывать отдельно/без double count.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_11_schema_sizes.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',11);

### Задание 12. `m_razhin.gpa_12_table_sizes`

**Что сделать:** Покажите крупнейшие учебные таблицы.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Строки, bytes, storage, policy.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_12_table_sizes.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',12);

### Задание 13. `m_razhin.gpa_13_segment_sizes`

**Что сделать:** Покажите размер выбранной таблицы по сегментам.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Ищите byte skew.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_13_segment_sizes.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',13);

### Задание 14. `m_razhin.gpa_14_skew`

**Что сделать:** Рассчитайте row/byte skew учебных таблиц.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Пустые segments тоже учитываются.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_14_skew.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',14);

### Задание 15. `m_razhin.gpa_15_bloat`

**Что сделать:** Оцените bloat/hidden tuples доступными метриками.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Метод зависит от heap/AO.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_15_bloat.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',15);

### Задание 16. `m_razhin.gpa_16_stats_age`

**Что сделать:** Покажите отсутствие/свежесть статистики.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

После load нужен ANALYZE.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_16_stats_age.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',16);

### Задание 17. `m_razhin.gpa_17_analyze`

**Что сделать:** Выполните ANALYZE учебной таблицы и зафиксируйте эффект.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Не запускайте cluster-wide без причины.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_17_analyze.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',17);

### Задание 18. `m_razhin.gpa_18_vacuum`

**Что сделать:** Выполните безопасный VACUUM учебной heap и сохраните метрики.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

VACUUM не аналог full rewrite.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_18_vacuum.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',18);

### Задание 19. `m_razhin.gpa_19_ao_stats`

**Что сделать:** Создайте VIEW AO auxiliary metadata.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

pg_appendonly, visimap/segfiles.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_19_ao_stats.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',19);

### Задание 20. `m_razhin.gpa_20_partitions`

**Что сделать:** Найдите excessive/small partitions.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Планирование страдает от тысяч leaf.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_20_partitions.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',20);

## Уровень 3 — resources, external health и incident response

### Задание 21. `m_razhin.gpa_21_resource_groups`

**Что сделать:** Создайте VIEW resource groups и limits.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Не изменяйте production-like defaults.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_21_resource_groups.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',21);

### Задание 22. `m_razhin.gpa_22_memory`

**Что сделать:** Покажите statement_mem/resource context текущей сессии.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Параметры действуют на разных уровнях.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_22_memory.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',22);

### Задание 23. `m_razhin.gpa_23_spill`

**Что сделать:** Создайте VIEW spill workfiles текущих/последних запросов.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Используйте gp_toolkit.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_23_spill.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',23);

### Задание 24. `m_razhin.gpa_24_cancel`

**Что сделать:** Безопасно отмените только созданный учебный запрос.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Сначала PID и ownership.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_24_cancel.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',24);

### Задание 25. `m_razhin.gpa_25_terminate`

**Что сделать:** Зафиксируйте отличие cancel и terminate.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Terminate закрывает session.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_25_terminate.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',25);

### Задание 26. `m_razhin.gpa_26_pxf`

**Что сделать:** Сводка PXF extension/server/path readiness.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Оба segment hosts.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_26_pxf.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',26);

### Задание 27. `m_razhin.gpa_27_hdfs`

**Что сделать:** Сводка NameNode/DataNode и персонального каталога.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Не изменяйте системные HDFS paths.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_27_hdfs.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',27);

### Задание 28. `m_razhin.gpa_28_incident`

**Что сделать:** Создайте incident report для медленного запроса.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Plan,skew,motion,spill,blocker,stats.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_28_incident.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',28);

### Задание 29. `m_razhin.gpa_29_runbook`

**Что сделать:** Создайте VIEW шагов runbook для типовых симптомов.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Симптом→проверка→безопасное действие→эскалация.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_29_runbook.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',29);

### Задание 30. `m_razhin.gpa_30_dashboard`

**Что сделать:** Создайте итоговый health dashboard курса.

Сначала выполните read-only диагностику. Любое действие укажите с точным target и сохраните before/after evidence.

<details><summary>Подсказка</summary>

Cluster,queries,storage,stats,PXF,HDFS.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpa_30_dashboard.

In [ ]:
%%sql
-- Ручная проверка диагностического результата.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('administration',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='administration' ORDER BY task_no;